# CH12 project 2 — a video through the filter, four ways

No camera in this one. A clip is decoded on the A53s, each frame is handed to
the Sobel filter, and the result goes both inline here and out to the
DisplayPort. The filter is available four ways — NumPy, OpenCV, and the PL in
any of its three implementations — and they are all driven through the same
register map, so switching between them is a different bitstream and not a
different notebook.

Two questions this notebook answers with numbers rather than adjectives:

1. **Is the PL bit-exact against the software reference?** The clip is
   generated, not filmed, so the expected answer is *zero differing samples* —
   not "close", not "looks right".
2. **What does each implementation cost per frame?** Which is what decides
   whether building the accelerator was worth it.

Pick the bitstream by editing `VARIANT` below: `sv`, `vhdl` or `hls`. All three
present the same register map, so nothing else in the notebook changes.

In [ ]:
import pathlib, sys

# sw/ holds the software reference and the driver. In the repo it is two levels
# up; on the board everything is normally copied into one directory.
for _cand in ("../../sw", "../sw", "sw", "."):
    if (pathlib.Path(_cand) / "sobel_ref.py").exists():
        sys.path.insert(0, str(pathlib.Path(_cand).resolve()))
        break
else:
    raise FileNotFoundError("cannot find sobel_ref.py -- copy sw/*.py next to this notebook")

print("sw/ ->", sys.path[0])

In [ ]:
import time
import numpy as np
from pynq import Overlay, allocate
from pynq.lib.video import VideoMode, DisplayPort, PIXEL_RGB

import sobel_ref as ref
import video_clip as vc
from filter_driver import VideoFilter, frame_address

VARIANT = "sv"          # sv | vhdl | hls
W, H = 1280, 720

def find_bitstream(name, variant=None):
    """Locate a bitstream in the repo layout or beside this notebook.

    In the repo the builds land in ../out_<variant>/; on the board everything
    is normally copied into one directory. Try both rather than making the
    reader edit a path.
    """
    cands = []
    if variant:
        cands.append(f"../out_{variant}/{name}")
    cands += [f"../out/{name}", name]
    for c in cands:
        if pathlib.Path(c).exists():
            return c
    raise FileNotFoundError(f"{name} not found -- looked in {cands}")

BITSTREAM = find_bitstream("video_sobel.bit", VARIANT)
print("overlay:", BITSTREAM)
ol = Overlay(BITSTREAM)
filt = VideoFilter(ol.video_filter_0)
print("modes:", ref.MODE_NAMES)

## The register map

Worth printing once. If `img_width` is not at 0x28 the accelerator will simply
never assert done, and that failure looks like a hung notebook rather than like
a wrong address.

In [ ]:
ol.video_filter_0.register_map

## A clip with a known answer

`video_clip.test_pattern` is a pure function of the frame index: a two-axis
colour gradient, three hard-edged bars for Sobel to find, a square that moves so
a repeated frame is visible, and the frame index packed into the top-left pixel.

It is generated rather than filmed for one reason. With real video there is no
way to say what *should* have come out, so a hardware-against-software
comparison can only ever be approximate. With this, it has an exact answer.

In [ ]:
import PIL.Image
from IPython.display import display

pattern = vc.test_pattern(W, H, 0)
print("frame index recovered from the top-left pixel:", vc.frame_index_of(pattern))
display(PIL.Image.fromarray(pattern[:, :, [2, 1, 0]]).resize((480, 270)))

## Buffers

`pynq.allocate` gives contiguous memory with a physical address the PL can
reach. Ordinary NumPy arrays cannot be handed to the accelerator at all — there
is no physical address to give it — which is what `frame_address` checks.

In [ ]:
src = allocate(shape=(H, W, 4), dtype=np.uint8)
dst = allocate(shape=(H, W, 4), dtype=np.uint8)
print("src @", hex(src.device_address))
print("dst @", hex(dst.device_address))

## The check that matters: is the PL bit-exact?

Every mode, against `sobel_ref.filter_frame`. The expected result is **0** in
every row. Anything else means the hardware and the software disagree about the
arithmetic, and the software is the one with unit tests.

In [ ]:
src[:] = vc.test_pattern(W, H, 3)
src.flush()

print(f"{'mode':>8}  {'PL ms':>7}  differing samples")
all_exact = True
for mode in ref.MODES:
    t = filt.run_frames(src, dst, mode)
    dst.invalidate()
    expected = ref.filter_frame(np.array(src), mode)
    bad = int(np.count_nonzero(np.array(dst) != expected))
    all_exact &= (bad == 0)
    print(f"{ref.MODE_NAMES[mode]:>8}  {t*1e3:7.2f}  {bad}")

print()
print("BIT-EXACT" if all_exact else "MISMATCH -- the PL and the reference disagree")

## What each implementation costs

The comparison is like for like only for NumPy, which is checked bit-exact
against the PL above. OpenCV is labelled approximate throughout because
`cvtColor` and `Sobel` use different coefficients, different rounding and a
replicated rather than zeroed border — but it is the fastest thing a competent
engineer would reach for on the A53s, so it is what the accelerator actually
has to beat.

In [ ]:
import statistics as st

# Median of many samples, not the mean of five. A five-rep mean of the OpenCV
# path once came back at 26 ms here where sixty samples say 39 -- small
# samples of a threaded library are not stable enough to build a claim on,
# and this is the claim the chapter rests on.
def timing(fn, n):
    fn()                                        # warm up
    xs = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); xs.append((time.perf_counter() - t0) * 1e3)
    return st.median(xs), min(xs), max(xs)

frame_np = np.array(src)
print(f"  {'mode':>8} {'PL (ms)':>22} {'OpenCV (ms)':>22} {'NumPy (ms)':>22}  {'PL vs OpenCV':>12}")
for mode in ref.MODES:
    pl  = timing(lambda: filt.run_frames(src, dst, mode), 40)
    cv_ = timing(lambda: ref.filter_frame_opencv(frame_np, mode), 20)
    npy = timing(lambda: ref.filter_frame(frame_np, mode), 10)
    fmt = lambda t: f"{t[0]:6.2f} ({t[1]:5.2f}-{t[2]:6.2f})"
    print(f"  {ref.MODE_NAMES[mode]:>8} {fmt(pl):>22} {fmt(cv_):>22} {fmt(npy):>22}  "
          f"{cv_[0]/pl[0]:11.1f}x")

### Reading that table

Expect the PL to be roughly the same for every mode — it is DDR-bound, moving
8 bytes per pixel whatever it does with them — while the software timings vary
a lot with how much arithmetic each mode needs. That is the whole shape of the
argument: **offloading pays in proportion to arithmetic per byte moved**, and
colour passthrough (mode 3, which does no arithmetic at all) is the control
that shows what the DDR round trip costs on its own.

## One copy, not none — and it dominates

The plan for this cell was to filter straight into the DisplayPort's own frame:
`dp.newframe()` hands back a buffer with a physical address, so the accelerator
could write it with nothing copied at all.

**The DisplayPort on this board will not have it.** It offers only 24bpp modes;
`VideoMode(W, H, 32)` is refused. The accelerator writes 4 bytes per pixel and
the display frame holds 3, so every frame needs a 32→24 bpp conversion on the
way out — and that conversion, not the filter, is what sets the frame rate.

The input side *is* still zero-copy: a clip frame, or a camera frame in
project 3, is read by the accelerator in place. `frame_address()` is what draws
the line, refusing the output frame rather than returning an address that would
shear the picture.

Two practical notes, both of which cost real time to find:

- **Use `cv2.cvtColor`, not NumPy slicing.** `frame[:] = dst[:, :, :3]` is a
  4→3 de-interleave that NumPy does element by element: about 368 ms a frame at
  720p. `cv2.cvtColor(..., COLOR_BGRA2BGR)` is the same work through NEON, and
  about 19 ms.
- **Nothing appears on screen while an X server is running.** PYNQ has to be
  DRM master to page-flip, and `pynq-x11.service` holds it. `sudo systemctl
  stop pynq-x11` first, or `writeframe` silently displays nothing while still
  returning promptly — which looks exactly like a working frame rate.

In [ ]:
# The DisplayPort on this board offers only 24bpp modes. Asking for 32 --
# VideoMode(W, H, 32) -- is refused outright with "is not supported", and
# handing configure() an EDID mode together with PIXEL_RGB yields a
# 3-channel view over a 4-byte stride, which is not something the accelerator
# can write in place. PIXEL_RGB is DRM_FORMAT_RGB888, whose memory order is
# B,G,R -- the same order the filter and the camera use.
dp = DisplayPort()
wanted = [m for m in dp.modes if m.width == W and m.height == H]
if not wanted:
    dp.close()
    raise RuntimeError(f"monitor does not offer {W}x{H}; "
                       f"it offers {sorted({(m.width, m.height) for m in dp.modes})}")
dp_mode = max(wanted, key=lambda m: m.fps)
dp.configure(dp_mode, PIXEL_RGB)
print(f"DisplayPort {dp_mode.width}x{dp_mode.height} @ {dp_mode.fps} Hz, "
      f"{dp_mode.bits_per_pixel} bpp")

probe = dp.newframe()
print("frame", probe.shape, "stride", probe.strides[0],
      "contiguous", probe.flags["C_CONTIGUOUS"])

# The accelerator writes 4 bytes per pixel and this frame holds 3, so it
# cannot be written in place. frame_address says so rather than handing back
# an address that would shear the picture.
try:
    frame_address(probe, W, H)
    print("usable in place: yes")
except ValueError as exc:
    print("usable in place: no --", str(exc).split(" -- ")[0])

# The conversion that costs. Write it the obvious way and it is ~368 ms a
# frame, because NumPy does the 4->3 de-interleave element by element.
# cv2.cvtColor is the same work through NEON, and about 19 ms.
import cv2
t0 = time.perf_counter()
for _ in range(10):
    cv2.cvtColor(np.asarray(dst), cv2.COLOR_BGRA2BGR, dst=probe)
print(f"32->24 bpp with cv2.cvtColor : {(time.perf_counter()-t0)/10*1e3:6.2f} ms")
t0 = time.perf_counter()
for _ in range(3):
    probe[:] = np.asarray(dst)[:, :, :3]
print(f"the same thing in NumPy      : {(time.perf_counter()-t0)/3*1e3:6.2f} ms")

## Play the clip

Two loops, timed separately: the accelerator filtering camera-sized frames
straight into the display, and the same work in software. The difference
between them is the answer to "was the accelerator worth building".

In [ ]:
import cv2
MODE = ref.MODE_SOBEL
N = 120

# Generate the clip ONCE, outside every timed loop. Synthesising a 720p test
# frame costs about 23 ms -- more than the filter does -- and charging that to
# all three loops equally does not make the comparison fair, it makes it
# meaningless: it buries the thing being compared under a constant that
# dominates it. A real application reads frames from a file or a sensor, not
# from NumPy.
CLIP = [vc.test_pattern(W, H, i) for i in range(30)]
buf = np.asarray(src)

def timed(label, body, n):
    body(0)                                    # warm up, then measure
    t0 = time.perf_counter()
    for i in range(n):
        body(i)
    fps = n / (time.perf_counter() - t0)
    print(f"  {label:<34} {fps:6.1f} fps")
    return fps

frame = [dp.newframe()]
def flip():
    dp.writeframe(frame[0]); frame[0] = dp.newframe()

def pl(i):
    buf[:] = CLIP[i % len(CLIP)]; src.flush()
    filt.run_frames(src, dst, MODE); dst.invalidate()
    cv2.cvtColor(np.asarray(dst), cv2.COLOR_BGRA2BGR, dst=frame[0]); flip()

def opencv(i):
    out = ref.filter_frame_opencv(CLIP[i % len(CLIP)], MODE)
    cv2.cvtColor(out, cv2.COLOR_BGRA2BGR, dst=frame[0]); flip()

def numpy_exact(i):
    out = ref.filter_frame(CLIP[i % len(CLIP)], MODE)
    cv2.cvtColor(out, cv2.COLOR_BGRA2BGR, dst=frame[0]); flip()

def unfiltered(i):
    cv2.cvtColor(np.ascontiguousarray(CLIP[i % len(CLIP)]),
                 cv2.COLOR_BGRA2BGR, dst=frame[0]); flip()

pl_fps  = timed("PL", pl, N)
cv_fps  = timed("OpenCV", opencv, N)
np_fps  = timed("NumPy (bit-exact)", numpy_exact, 30)
raw_fps = timed("no filter at all", unfiltered, N)

print(f"\n  PL vs OpenCV {pl_fps/cv_fps:.2f}x    PL vs NumPy {pl_fps/np_fps:.1f}x")
print(f"  PL reaches {pl_fps/raw_fps*100:.0f}% of the no-filter ceiling")
print("\n  These are vsync-locked: writeframe blocks on the page flip, so the")
print("  rates land near 60/n rather than on the raw loop time.")

Both loops include generating the source frame, which is not free and is charged
to both equally. What differs between them is only where the filtering happens.

## Your own video

Point `VIDEO` at any file OpenCV can open. `read_clip` letterboxes to the size
you ask for and hands back BGRA frames — the same layout the camera produces, so
the filter cannot tell the two apart. That is what makes project 3 possible.

In [ ]:
VIDEO = None      # e.g. "clip.mp4"

if VIDEO:
    out_frames = []
    for i, f in enumerate(vc.read_clip(VIDEO, W, H, max_frames=90)):
        src[:] = f
        src.flush()
        filt.run_frames(src, dst, ref.MODE_SOBEL)
        dst.invalidate()
        out_frames.append(np.array(dst))
    vc.write_clip("filtered.mp4", out_frames, fps=30)
    print(f"wrote filtered.mp4 ({len(out_frames)} frames)")
else:
    print("set VIDEO to a file path to run this cell")

In [ ]:
from IPython.display import Video
Video("filtered.mp4", embed=True, width=640) if VIDEO else "no video written"

## Clean up

In [ ]:
dp.close()
src.freebuffer()
dst.freebuffer()
print("released")